# Agent Memory Server - Interactive Technical Guide

**Date:** February 5, 2026

This notebook provides a comprehensive, interactive walkthrough of the Redis Agent Memory Server, focusing on **how to integrate memory into your AI applications**.

---

## Prerequisites

Before running this notebook, ensure you have:

1. **Redis 8 running** (via Docker):
   ```bash
   docker-compose up redis -d
   ```

2. **Agent Memory Server running** (development mode):
   ```bash
   uv run agent-memory api --task-backend=asyncio
   ```

3. **Environment variables set**:
   ```bash
   export OPENAI_API_KEY=your-key-here
   export DISABLE_AUTH=true  # For development only
   ```

4. **Python dependencies installed**:
   ```bash
   pip install agent-memory-client httpx openai
   ```

---

## Section 1: The Problem & Solution

### The Problem
LLMs are **stateless** - they don't remember past conversations. Every interaction starts fresh, losing valuable context about user preferences, past decisions, and conversation history.

### The Solution
The Agent Memory Server provides a dedicated memory layer for AI agents with:
- **Working Memory**: Session-scoped, short-term storage for current conversations
- **Long-term Memory**: Persistent storage with semantic search capabilities

### Two-Tier Architecture
```
Working Memory (Session-scoped)  →  Long-term Memory (Persistent)
    ↓                                      ↓
- Messages                         - Semantic search
- Structured memories              - Topic modeling  
- Summary of past messages         - Entity recognition
- Metadata                         - Deduplication
```

---

## Section 2: Quick Start - Setup

Let's get connected to the memory server.

In [1]:
# Import required libraries
import asyncio
import json
import os
from datetime import datetime, timezone

import httpx
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Configuration
BASE_URL = "http://localhost:8000"
NAMESPACE = "travel_agent"  # Same namespace as the travel agent example

print(f"Base URL: {BASE_URL}")
print(f"Namespace: {NAMESPACE}")
print(f"OpenAI API Key: {'✓ loaded' if os.environ.get('OPENAI_API_KEY') else '✗ not found'}")

Base URL: http://localhost:8000
Namespace: travel_agent
OpenAI API Key: ✓ loaded


In [2]:
# Health check
async def health_check():
    async with httpx.AsyncClient(base_url=BASE_URL) as client:
        response = await client.get("/v1/health")
        response.raise_for_status()
        return response.json()

health = await health_check()
print(f"Server is healthy!")
print(f"Server timestamp: {datetime.fromtimestamp(health['now'] / 1000)}")

Server is healthy!
Server timestamp: 2026-02-05 10:10:41.230000


In [4]:
# Initialize the Python SDK client
from agent_memory_client import MemoryAPIClient, MemoryClientConfig

config = MemoryClientConfig(
    base_url=BASE_URL,
    timeout=30.0,
    default_namespace=NAMESPACE  # "travel_agent" - same as the travel agent example
)

client = MemoryAPIClient(config)
print(f"Memory client initialized!")
print(f"Default namespace: {config.default_namespace}")

Memory client initialized!
Default namespace: travel_agent


## 🔍 What's Happening Behind the Scenes in Redis?

As you run this notebook, data is being stored in Redis. Here's what the data structure looks like (as seen in Redis Insight):

### Redis Key Structure

```
📁 memory_idx/                    # Long-term memory index (vector embeddings)
   └── HASH  01KGNPMZQF70S3...    # Each memory record (~9KB each)
                                   # Contains: text, embedding vector, metadata
   
📁 memory-server/                 # Background task queue (Docket)
   └── 📁 runs/                   # Task execution history
   └── STREAM  stream             # Task stream for workers

📁 sessions/                      # Session tracking
   └── SORTED SET  travel_agent   # All sessions in this namespace
                                   # Sorted by last activity time

📁 working_memory/                # Session-scoped conversation memory
   └── 📁 travel_agent/           # Namespace
       └── 📁 nitin/              # User ID
           └── JSON  nitin-travel-session  # The actual session data (~944B)
                                            # Contains: messages, context, metadata
```

### Key Types Explained

| Type | Purpose | Example |
|------|---------|---------|
| **HASH** | Long-term memory records with vector embeddings | `memory_idx:01KGN...` |
| **JSON** | Working memory sessions (conversations) | `working_memory:travel_agent:nitin:nitin-travel-session` |
| **SORTED SET** | Session index for fast lookup by namespace | `sessions:travel_agent` |
| **STREAM** | Background task queue for async processing | `memory-server:stream` |

### Data Flow

1. **`create_long_term_memory()`** → Creates HASH entries in `memory_idx/`
2. **`put_working_memory()`** → Creates JSON entry in `working_memory/`
3. **`memory_prompt()`** → Reads from both and combines them
4. **Background tasks** → Process via `memory-server/stream`

> 💡 **Tip**: Open Redis Insight at `http://localhost:16381` to explore the data in real-time as you run cells!

---

## Section 3: The Three Integration Patterns

The most common question is: **"How do I actually get memories into and out of my LLM?"**

There are three distinct patterns, each optimized for different use cases:

| Pattern | Who Decides | Best For | Memory Flow |
|---------|-------------|----------|-------------|
| **Code-Driven (SDK)** | Your code | Apps, workflows, deterministic behavior | `Code → SDK → Memory` |
| **LLM-Driven (Tools)** | The LLM | Conversational agents, chatbots | `LLM ↔ Tools ↔ Memory` |
| **Background Extraction** | Automatic | Learning systems, passive accumulation | `Conversation → Auto Extract → Memory` |

### Decision Guide

```
Start here: Do you need predictable, deterministic memory behavior?
    │
    ├── YES → Use Pattern 1: Code-Driven (SDK)
    │         Your code explicitly stores/retrieves memories
    │
    └── NO → Do you want the LLM to decide what to remember?
              │
              ├── YES → Use Pattern 2: LLM-Driven (Tools)
              │         LLM uses memory tools during conversation
              │
              └── NO → Use Pattern 3: Background Extraction
                       System automatically learns from conversations
```

**Pro tip**: These patterns are NOT mutually exclusive! Most production systems combine them.

---

## Section 4: Pattern 1 - Code-Driven (SDK)

### When to Use
- You need **predictable, deterministic** memory behavior
- Building applications, workflows, or pipelines
- You know exactly when to store/retrieve memories

### How It Works

```
┌─────────────────────────────────────────────────────────────────┐
│                     YOUR APPLICATION CODE                        │
├─────────────────────────────────────────────────────────────────┤
│                                                                  │
│   1. User sends message                                          │
│          ↓                                                       │
│   2. YOUR CODE calls memory_prompt() to get context              │
│          ↓                                                       │
│   3. Send enriched prompt to LLM                                 │
│          ↓                                                       │
│   4. LLM generates response                                      │
│          ↓                                                       │
│   5. YOUR CODE decides what to store                             │
│          ↓                                                       │
│   6. Call create_long_term_memory() or put_working_memory()      │
│                                                                  │
└─────────────────────────────────────────────────────────────────┘
```

**Key Point**: YOUR CODE makes all memory decisions, not the LLM.

### Example: Travel Agent (Code-Driven)
See: `examples/travel_agent.py`

In [5]:
# Pattern 1 Code driven

from agent_memory_client.models import (
    MemoryMessage, 
    WorkingMemory, 
    ClientMemoryRecord, 
    MemoryTypeEnum
)

# Use "nitin" as our demo user - matching the travel agent example
SESSION_ID = "nitin-travel-session"
USER_ID = "nitin"
NAMESPACE = "travel_agent"  # Same namespace as the travel agent

# Step 1: Create/get a working memory session for Nitin
created, working_memory = await client.get_or_create_working_memory(
    session_id=SESSION_ID,
    namespace=NAMESPACE,
    user_id=USER_ID
)

print(f"Session {'created' if created else 'already existed'}: {SESSION_ID}")
print(f"User: {USER_ID}")
print(f"Namespace: {NAMESPACE}")

Session created: nitin-travel-session
User: nitin
Namespace: travel_agent


In [6]:
# Diff session
from agent_memory_client.models import (
    MemoryMessage,
    WorkingMemory,
    ClientMemoryRecord,
    MemoryTypeEnum
)

# Use "nitin" as our demo user - matching the travel agent example
SESSION_ID = "nitin-travel-session-2"
USER_ID = "nitin"
NAMESPACE = "travel_agent"  # Same namespace as the travel agent

# Step 1: Create/get a working memory session for Nitin
created, working_memory = await client.get_or_create_working_memory(
    session_id=SESSION_ID,
    namespace=NAMESPACE,
    user_id=USER_ID
)

print(f"Session {'created' if created else 'already existed'}: {SESSION_ID}")
print(f"User: {USER_ID}")
print(f"Namespace: {NAMESPACE}")

Session created: nitin-travel-session-2
User: nitin
Namespace: travel_agent


### What does this mean?

If We Create Two Sessions (e.g., nitin-travel-session and nitin-travel-session-2)

Each session is completely independent. They create separate Redis keys:
```
📁 working_memory/
   └── 📁 travel_agent/           # Same namespace
       └── 📁 nitin/              # Same user
           ├── JSON  nitin-travel-session    # Session 1 (separate conversation)
           └── JSON  nitin-travel-session-2  # Session 2 (separate conversation)
```
- Each session has its own conversation history (messages)
- Each session has its own context window (summarization happens independently)
- Sessions are like separate chat threads for the same user
- Use case: User opens a new chat window, starts a new trip planning conversation

If We Create Two Namespaces (e.g., travel_agent and customer_support)
```
📁 working_memory/
   ├── 📁 travel_agent/           # Namespace 1
   │   └── 📁 nitin/
   │       └── JSON  nitin-session-1
   │
   └── 📁 customer_support/       # Namespace 2 (completely separate)
       └── 📁 nitin/
           └── JSON  nitin-session-1   # Same session ID, but different namespace!

📁 memory_idx/
   ├── HASH  01KGN... (namespace=travel_agent)     # Only searchable within travel_agent
   └── HASH  01KGP... (namespace=customer_support) # Only searchable within customer_support
```

- Same user can have different memories per namespace
 - Searches are scoped to namespace - travel memories don't leak into support
- Use case: Multi-tenant apps, different AI agents, different product areas


NOTE:
Working Memory: Scoped by Namespace + User + Session
```working_memory:{namespace}:{user_id}:{session_id}```

So the same session ID in different namespaces are completely separate:
```
working_memory:travel_agent:nitin:session-1    # One conversation
working_memory:customer_support:nitin:session-1  # Completely different conversation
```

Long-Term Memory: Scoped by Namespace + User
Each memory record has namespace and user_id as metadata fields:
```
memory_idx:01KGNPMZQF70S3...
  ├── text: "Nitin prefers vegetarian food"
  ├── namespace: "travel_agent"      ← Scoped to this namespace
  ├── user_id: "nitin"               ← Scoped to this user
  └── vector: [...]
```

When you search, you filter by namespace:
```python
results = await client.search_long_term_memory(
    text="food preferences",
    namespace={"eq": "travel_agent"},  # Only searches THIS namespace
    user_id={"eq": "nitin"}
)
```

```
┌─────────────────────────────────────────────────────────────────┐
│                    NAMESPACE: travel_agent                       │
├─────────────────────────────────────────────────────────────────┤
│  WORKING MEMORY                                                  │
│  ├── nitin:nitin-travel-session     (Japan trip conversation)   │
│  └── nitin:nitin-travel-session-2   (Italy trip conversation)   │
│                                                                  │
│  LONG-TERM MEMORY                                                │
│  ├── "Nitin prefers vegetarian food"                            │
│  ├── "Nitin likes hiking"                                       │
│  └── "Nitin prefers mid-tier hotels"                            │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│                  NAMESPACE: customer_support                     │
├─────────────────────────────────────────────────────────────────┤
│  WORKING MEMORY                                                  │
│  └── nitin:support-ticket-123       (Support conversation)       │
│                                                                  │
│  LONG-TERM MEMORY                                                │
│  ├── "Nitin had billing issue in January"                       │
│  └── "Nitin prefers email over phone"                           │
│                                                                  │
│  ⚠️ CANNOT see travel_agent memories!                           │
└─────────────────────────────────────────────────────────────────┘
```


### Step 2 Seed data into longterm memory


In [7]:

# Nitin's travel preferences (from the travel agent demo)
memories_to_store = [
    ClientMemoryRecord(
        text="Comfort travel - middle tier (not luxury, not budget)",
        memory_type=MemoryTypeEnum.SEMANTIC,
        topics=["travel", "preferences"],
        user_id=USER_ID,
        namespace=NAMESPACE
    ),
    ClientMemoryRecord(
        text="Vegetarian diet - needs vegetarian restaurant options",
        memory_type=MemoryTypeEnum.SEMANTIC,
        topics=["travel", "preferences"],
        user_id=USER_ID,
        namespace=NAMESPACE
    ),
    ClientMemoryRecord(
        text="Extra leg room on flights (premium economy or exit row and budget allows, anything goes)",
        memory_type=MemoryTypeEnum.SEMANTIC,
        topics=["travel", "preferences"],
        user_id=USER_ID,
        namespace=NAMESPACE
    ),
    ClientMemoryRecord(
        text="Prefers hotels with good amenities",
        memory_type=MemoryTypeEnum.SEMANTIC,
        topics=["travel", "preferences"],
        user_id=USER_ID,
        namespace=NAMESPACE
    ),
    ClientMemoryRecord(
        text="Enjoys technology, sports, outdoords, and innovation hubs",
        memory_type=MemoryTypeEnum.SEMANTIC,
        topics=["travel", "preferences"],
        user_id=USER_ID,
        namespace=NAMESPACE
    )
]

print(f"Storing {len(memories_to_store)} preferences for user '{USER_ID}'...")
result = await client.create_long_term_memory(memories_to_store)
print(f"Result: {result.status}")

# Show what we just stored
print("\nStored memories:")
for i, mem in enumerate(memories_to_store, 1):
    print(f"  {i}. {mem.text}")

Storing 5 preferences for user 'nitin'...
Result: ok

Stored memories:
  1. Comfort travel - middle tier (not luxury, not budget)
  2. Vegetarian diet - needs vegetarian restaurant options
  3. Extra leg room on flights (premium economy or exit row and budget allows, anything goes)
  4. Prefers hotels with good amenities
  5. Enjoys technology, sports, outdoords, and innovation hubs


### Step 3: Retrieve relevant context using memory_prompt()

Explain memory_prompt()

Where does Data comes from?
   

In [8]:

import time
time.sleep(2)  # Wait for indexing (memories need to be embedded)

user_query = "I'm planning a trip to Japan. What should I know about my preferences?"

print(f"Query: '{user_query}'")
print(f"Searching memories for user: {USER_ID}\n")

# Note: The memory_prompt() endpoint is designed to return ready-to-use messages for an LLM.
# The memories are pre-formatted into the system prompt so you can directly
# pass messages to OpenAI/Claude without additional processing.
prompt_result = await client.memory_prompt(
    session_id=SESSION_ID,
    query=user_query,
    namespace=NAMESPACE,
    user_id=USER_ID,
    long_term_search={
        "limit": 5,
        "distance_threshold": 0.7,
        "user_id": {"eq": USER_ID}  # Only search Nitin's memories
    }
)




Query: 'I'm planning a trip to Japan. What should I know about my preferences?'
Searching memories for user: nitin



In [9]:
prompt_result

{'messages': [{'role': 'system',
   'content': {'type': 'text',
    'text': "## Long term memories related to the user's query\n - Prefers hotels with good amenities (ID: 01KGQ65EQJ3YQ0HJ6PJ9AT3Y4C)\n- Comfort travel - middle tier (not luxury, not budget) (ID: 01KGQ65EQHP04EFHB65YZ536ED)",
    'annotations': None,
    '_meta': None}},
  {'role': 'user',
   'content': {'type': 'text',
    'text': "I'm planning a trip to Japan. What should I know about my preferences?",
    'annotations': None,
    '_meta': None}}]}

In [10]:
messages = prompt_result.get("messages", [])

print("=" * 60)
print("MEMORY PROMPT RESULT")
print("=" * 60)
print(f"Messages in context: {len(messages)}")

# Count long-term memories from the system message
# The memory_prompt() API embeds memories directly in the system message text
# Each memory line starts with " - " (space-dash-space)
memory_count = 0
for msg in messages:
    if msg.get("role") == "system":
        content = msg.get("content", {})
        if isinstance(content, dict):
            text = content.get("text", "")
        else:
            text = str(content)
        # Count lines that start with " - " (memory entries)
        memory_count = text.count("\n - ") + (1 if text.strip().startswith("- ") else 0)
        break

print(f"Long-term memories found: {memory_count}")
print()

# Display the messages
for msg in messages:
    role = msg.get("role", "unknown")
    content = msg.get("content", {})
    if isinstance(content, dict):
        text = content.get("text", str(content))
    else:
        text = str(content)
    print(f"[{role.upper()}]")
    print(text)
    print()

messages

MEMORY PROMPT RESULT
Messages in context: 2
Long-term memories found: 1

[SYSTEM]
## Long term memories related to the user's query
 - Prefers hotels with good amenities (ID: 01KGQ65EQJ3YQ0HJ6PJ9AT3Y4C)
- Comfort travel - middle tier (not luxury, not budget) (ID: 01KGQ65EQHP04EFHB65YZ536ED)

[USER]
I'm planning a trip to Japan. What should I know about my preferences?



[{'role': 'system',
  'content': {'type': 'text',
   'text': "## Long term memories related to the user's query\n - Prefers hotels with good amenities (ID: 01KGQ65EQJ3YQ0HJ6PJ9AT3Y4C)\n- Comfort travel - middle tier (not luxury, not budget) (ID: 01KGQ65EQHP04EFHB65YZ536ED)",
   'annotations': None,
   '_meta': None}},
 {'role': 'user',
  'content': {'type': 'text',
   'text': "I'm planning a trip to Japan. What should I know about my preferences?",
   'annotations': None,
   '_meta': None}}]

### Step 4: Store conversation in working memory

Working memory stores the current conversation for this session.
This is separate from long-term memory (which stores facts/preferences).

In [11]:

messages = [
    MemoryMessage(role="user", content="I'm planning a trip to Japan next month!"),
    MemoryMessage(role="assistant", content="Exciting! Based on your preferences, I know you enjoy hiking and vegetarian food. Japan has amazing options for both!"),
    MemoryMessage(role="user", content="Yes! I'd love to hike Mount Fuji and find good vegetarian ramen."),
    MemoryMessage(role="assistant", content="Perfect! I'll remember your interest in Mount Fuji. For vegetarian ramen, Kyoto has excellent options.")
]

updated_memory = WorkingMemory(
    session_id=SESSION_ID,
    namespace=NAMESPACE,
    messages=messages,
    user_id=USER_ID
)

response = await client.put_working_memory(SESSION_ID, updated_memory)
print(f"Working memory updated with {len(messages)} messages")
print(f"Session: {SESSION_ID}")
response

MemoryMessage created without explicit created_at timestamp. This will become required in a future version. Please provide created_at for accurate message ordering.


Working memory updated with 4 messages
Session: nitin-travel-session-2


WorkingMemoryResponse(messages=[MemoryMessage(role='user', content="I'm planning a trip to Japan next month!", id='01KGQ6F7Q3AB0GHS96FG254WGC', created_at=datetime.datetime(2026, 2, 5, 15, 24, 28, 771507, tzinfo=TzInfo(0)), persisted_at=None, discrete_memory_extracted='f'), MemoryMessage(role='assistant', content='Exciting! Based on your preferences, I know you enjoy hiking and vegetarian food. Japan has amazing options for both!', id='01KGQ6F7Q3AB0GHS96FG254WGD', created_at=datetime.datetime(2026, 2, 5, 15, 24, 28, 771539, tzinfo=TzInfo(0)), persisted_at=None, discrete_memory_extracted='f'), MemoryMessage(role='user', content="Yes! I'd love to hike Mount Fuji and find good vegetarian ramen.", id='01KGQ6F7Q3AB0GHS96FG254WGE', created_at=datetime.datetime(2026, 2, 5, 15, 24, 28, 771553, tzinfo=TzInfo(0)), persisted_at=None, discrete_memory_extracted='f'), MemoryMessage(role='assistant', content="Perfect! I'll remember your interest in Mount Fuji. For vegetarian ramen, Kyoto has excellen

In [15]:
response.messages[-1]

MemoryMessage(role='assistant', content="Perfect! I'll remember your interest in Mount Fuji. For vegetarian ramen, Kyoto has excellent options.", id='01KGQ6F7Q3AB0GHS96FG254WGF', created_at=datetime.datetime(2026, 2, 5, 15, 24, 28, 771565, tzinfo=TzInfo(0)), persisted_at=None, discrete_memory_extracted='f')

In [16]:
# Let's search and see what's actually stored in Redis

# Search all memories for this user
all_memories = await client.search_long_term_memory(
    text="travel preferences",  # Broad search
    namespace={"eq": "travel_agent"},
    user_id={"eq": "nitin"},
    limit=20
)

print(f"\nFound {all_memories.total} memories:\n")

for idx, memory in enumerate(all_memories.memories, 1):
    # Calculate relevance score (1 - distance)
    relevance = (1 - memory.dist) * 100 if memory.dist else 0
    
    print(f"{idx}. [{relevance:.0f}% relevant]")
    print(f"   ID: {memory.id}")
    print(f"   Text: {memory.text}")
    print(f"   Topics: {memory.topics}")
    print(f"   Created: {memory.created_at}\n")


Found 9 memories:

1. [48% relevant]
   ID: 01KGQ65EQHP04EFHB65YZ536ED
   Text: Comfort travel - middle tier (not luxury, not budget)
   Topics: ['travel', 'preferences']
   Created: 2026-02-05 15:19:08.273890+00:00

2. [41% relevant]
   ID: 01KGQ65EQJ3YQ0HJ6PJ9AT3Y4B
   Text: Extra leg room on flights (premium economy or exit row and budget allows, anything goes)
   Topics: ['travel', 'preferences']
   Created: 2026-02-05 15:19:08.274876+00:00

3. [37% relevant]
   ID: 01KGQ65EQJ3YQ0HJ6PJ9AT3Y4C
   Text: Prefers hotels with good amenities
   Topics: ['travel', 'preferences']
   Created: 2026-02-05 15:19:08.274893+00:00

4. [34% relevant]
   ID: 01KGQ6FE0809B98WPX0V0AAY5J
   Text: User enjoys hiking and vegetarian food
   Topics: ['travel', 'food', 'hiking']
   Created: 2026-02-05 15:24:35.208709+00:00

5. [33% relevant]
   ID: 01KGQ6FE0809B98WPX0V0AAY5K
   Text: User is planning a trip to Japan in March 2026
   Topics: ['travel', 'Japan']
   Created: 2026-02-05 15:24:35.208786+00:00


### Pattern 1 Summary

**Where memory operations happen:**
- `memory_prompt()` - Called by YOUR CODE before LLM call
- `create_long_term_memory()` - Called by YOUR CODE after conversation
- `put_working_memory()` - Called by YOUR CODE to save conversation

**Advantages:**
- Predictable, deterministic behavior
- No token overhead for tool schemas
- Full control over what gets stored
- Easier to debug and test

**Disadvantages:**
- Requires explicit logic for memory decisions
- May miss nuanced information the LLM would catch
- More code to maintain

bb---

## Section 5: Pattern 2 - LLM-Driven (Tool-Based)

### When to Use
- Building **conversational agents or chatbots**
- You want the LLM to decide what's worth remembering
- Users might explicitly ask to "remember this" or "what did I say about..."

### How It Works

```
┌─────────────────────────────────────────────────────────────────┐
│                         LLM DECIDES                              │
├─────────────────────────────────────────────────────────────────┤
│                                                                  │
│   1. User sends message                                          │
│          ↓                                                       │
│   2. Send to LLM WITH memory tool schemas                        │
│          ↓                                                       │
│   3. LLM DECIDES: "Should I store/search memory?"                │
│          ↓                                                       │
│   4. If yes → LLM returns tool_calls                             │
│          ↓                                                       │
│   5. YOUR CODE executes tool calls via resolve_function_call()   │
│          ↓                                                       │
│   6. Return results to LLM for final response                    │
│                                                                  │
└─────────────────────────────────────────────────────────────────┘
```

**Key Point**: The LLM decides when to use memory, your code just executes.

### Example: AI Tutor (LLM-Driven)
See: `examples/ai_tutor.py`, `examples/langchain_integration_example.py`

In [17]:
# PATTERN 2: LLM-Driven Memory (Tool-Based)
# The LLM decides when to store/retrieve memories using tools

# Step 1: Get memory tool schemas for the LLM
tools = MemoryAPIClient.get_all_memory_tool_schemas()

print(f"Available memory tools for LLM:\n\n{tools}\n\n")
for tool in tools.to_list():
    print(f"  - {tool['function']['name']}: {tool['function']['description']}\n")

Available memory tools for LLM:

ToolSchemaCollection(['search_memory', 'get_or_create_working_memory', 'lazily_create_long_term_memory', 'update_working_memory_data', 'get_long_term_memory', 'eagerly_create_long_term_memory', 'edit_long_term_memory', 'delete_long_term_memories', 'get_current_datetime'])


  - search_memory: Search long-term memory for relevant information using semantic vector search. Use this when you need to find previously stored information about the user, such as their preferences, past conversations, or important facts. Examples: 'Find information about user food preferences', 'What did they say about their job?', 'Look for travel preferences'. This searches only long-term memory, not current working memory - use get_working_memory for current session info. IMPORTANT: The result includes 'memories' with an 'id' field; use these IDs when calling edit_long_term_memory or delete_long_term_memories.

  - get_or_create_working_memory: Get the current working memory s

In [18]:
# Step 2: Send tools to LLM with the conversation
# The LLM will decide whether to use memory tools

import openai
import os

# Check if OpenAI API key is available
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    print("⚠️  OPENAI_API_KEY not set - LLM-driven pattern cells will be skipped")
    print("   Set it with: export OPENAI_API_KEY=your-key-here")
    openai_client = None
else:
    openai_client = openai.AsyncOpenAI()
    print("✓ OpenAI client initialized")

SESSION_ID_LLM = "nitin-llm-session"
USER_ID_LLM = "nitin"

# System prompt that encourages memory usage
system_prompt = """You are a helpful assistant with persistent memory capabilities.

When to remember (use create_long_term_memories tool):
- User preferences (food, communication style, etc.)
- Important personal information
- Project details and context

When to search memory (use search_long_term_memory tool):
- User asks about previous conversations
- Context would help provide better responses
- User references something from the past

Always be transparent about what you're remembering."""

user_message = "Hi! I'm Nitin. I'm planning a trip to Tokyo and want to find good vegetarian restaurants."

print(f"User: {user_message}")
print("\nSending to LLM with memory tools...")

✓ OpenAI client initialized
User: Hi! I'm Nitin. I'm planning a trip to Tokyo and want to find good vegetarian restaurants.

Sending to LLM with memory tools...


In [19]:
# Step 3: Call LLM with tools - LLM DECIDES whether to use them

response = await openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message}
    ],
    tools=tools.to_list()  # Pass memory tools to LLM
)

# Check if LLM decided to use any tools
message = response.choices[0].message

if message.tool_calls:
    print(f"LLM decided to use {len(message.tool_calls)} tool(s):")
    for tc in message.tool_calls:
        print(f"  - {tc.function.name}")
        print(f"    Args: {tc.function.arguments}")
else:
    print("LLM decided NOT to use any memory tools")
    print(f"Response: {message.content}")

LLM decided to use 1 tool(s):
  - lazily_create_long_term_memory
    Args: {"text":"User's name is Nitin and is planning a trip to Tokyo.","memory_type":"semantic","topics":["travel"],"entities":["Tokyo"]}


In [20]:
# Step 4: Execute tool calls using resolve_function_call()
# Our code executes what the LLM decided

tool_results = []
    
for tool_call in message.tool_calls:
    print(f"\nExecuting: {tool_call.function.name}")

    # resolve_function_call() handles all memory operations
    # Returns a TypedDict with success, function_name, result, error, formatted_response
    result = await client.resolve_function_call(
        function_name=tool_call.function.name,
        function_arguments=json.loads(tool_call.function.arguments),
        session_id=SESSION_ID_LLM,
        namespace=NAMESPACE,
        user_id=USER_ID_LLM
    )

    formatted = result.get("formatted_response", str(result))
    print(f"  Result: {formatted}")
    tool_results.append({
        "tool_call_id": tool_call.id,
        "role": "tool",
        "content": json.dumps(result.get("formatted_response", result))
    })

print(f"\nExecuted {len(tool_results)} tool call(s)")


Executing: lazily_create_long_term_memory
  Result: Successfully stored semantic memory: User's name is Nitin and is planning a trip to Tok...

Executed 1 tool call(s)


In [21]:
result

{'success': True,
 'function_name': 'lazily_create_long_term_memory',
 'result': {'success': True,
  'memory_type': 'semantic',
  'text_preview': "User's name is Nitin and is planning a trip to Tokyo.",
  'topics': ['travel'],
  'entities': ['Tokyo'],
  'summary': "Successfully stored semantic memory: User's name is Nitin and is planning a trip to Tok..."},
 'error': None,
 'formatted_response': "Successfully stored semantic memory: User's name is Nitin and is planning a trip to Tok..."}

In [22]:
# Step 5: Send tool results back to LLM for final response

# Build messages with tool results
messages_with_results = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_message},
    {"role": "assistant", "content": None, "tool_calls": [
        {"id": tc.id, "type": "function", "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
        for tc in message.tool_calls
    ]},
] + tool_results

# Get final response
final_response = await openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages_with_results
)

print("Final LLM Response:")
print(f"  {final_response.choices[0].message.content}")

Final LLM Response:
  Hi Nitin! That sounds exciting! For vegetarian restaurants in Tokyo, here are a few recommendations:

1. **Naritake** - This restaurant offers a variety of vegetarian ramen options.
2. **T's Tantan** - Located in Tokyo Station, this place is famous for its vegan ramen.
3. **Ain Soph. Journey** - A completely vegan restaurant with a menu full of delicious dishes.
4. **Kushitei** - A great spot for vegetarian skewers and side dishes.
5. **Sushi Saito** - Offers a vegetarian sushi menu.

Let me know if you need more information or specific recommendations!


In [38]:
messages_with_results

[{'role': 'system',
  'content': "You are a helpful assistant with persistent memory capabilities.\n\nWhen to remember (use create_long_term_memories tool):\n- User preferences (food, communication style, etc.)\n- Important personal information\n- Project details and context\n\nWhen to search memory (use search_long_term_memory tool):\n- User asks about previous conversations\n- Context would help provide better responses\n- User references something from the past\n\nAlways be transparent about what you're remembering."},
 {'role': 'user',
  'content': "Hi! I'm Nitin. I'm planning a trip to Tokyo and want to find good vegetarian restaurants."},
 {'role': 'assistant',
  'content': None,
  'tool_calls': [{'id': 'call_Ux7XEoqyJxtOl68MPlE9ARjZ',
    'type': 'function',
    'function': {'name': 'lazily_create_long_term_memory',
     'arguments': '{"text":"User\'s name is Nitin and is planning a trip to Tokyo.","memory_type":"semantic","topics":["travel"],"entities":["Tokyo"]}'}}]},
 {'tool_

### Pattern 2 Summary

**Where memory operations happen:**
- `get_all_memory_tool_schemas()` - Called ONCE to get tool definitions
- LLM decides - Returns `tool_calls` when it wants to use memory
- `resolve_function_call()` - Called by YOUR CODE to execute LLM's decisions

**Advantages:**
- Natural conversation flow
- LLM understands context and nuance
- Users can explicitly ask to remember/recall
- Flexible and adaptive
- Seems to be moving towards this direction (Example ADK -> less control)

**Disadvantages:**
- Token overhead for tool schemas
- Inconsistent behavior (LLM might not always use memory)
- More API calls and latency
- Harder to debug

---

## Section 6: Pattern 3 - Background Extraction

### When to Use
- You want the system to **automatically learn** from conversations
- Building learning systems that improve over time
- You don't want to manually decide what to remember

### How It Works

```
┌─────────────────────────────────────────────────────────────────┐
│                    AUTOMATIC EXTRACTION                          │
├─────────────────────────────────────────────────────────────────┤
│                                                                  │
│   1. Conversation happens normally                               │
│          ↓                                                       │
│   2. Store conversation in working memory                        │
│          ↓                                                       │
│   3. SYSTEM AUTOMATICALLY:                                       │
│      - Analyzes conversation for important info                  │
│      - Extracts structured memories                              │
│      - Applies contextual grounding                              │
│      - Deduplicates similar memories                             │
│      - Stores in long-term memory                                │
│                                                                  │
└─────────────────────────────────────────────────────────────────┘
```

**Key Point**: Just store conversations, the system learns automatically.

### Extraction Strategies

| Strategy | Purpose | Use Case |
|----------|---------|----------|
| **discrete** (default) | Extract individual facts | General-purpose |
| **summary** | Condense entire conversations | Meeting notes |
| **user_preferences** | Target user settings/traits | Personalization |
| **custom** | Your own extraction logic | Specialized domains |

In [23]:
# PATTERN 3: Background Extraction
# Just store conversations - system extracts memories automatically

from agent_memory_client.models import MemoryStrategyConfig

SESSION_ID_AUTO = "nitin-auto-session"
USER_ID_AUTO = "nitin"

# Step 1: Create session WITH extraction strategy configured
created, working_memory = await client.get_or_create_working_memory(
    session_id=SESSION_ID_AUTO,
    namespace=NAMESPACE,
    # Configure automatic extraction
    long_term_memory_strategy=MemoryStrategyConfig(
        strategy="discrete"  # Extract individual facts (default)
    )
)

print(f"Session created with automatic extraction: {SESSION_ID_AUTO}")
strategy = working_memory.long_term_memory_strategy
print(f"Strategy: {strategy.strategy if strategy else 'default'}")

Session created with automatic extraction: nitin-auto-session
Strategy: discrete


In [24]:
# Step 2: Just store the conversation - extraction happens automatically!
conversation = [
    MemoryMessage(role="user", content="I'm Nitin. I'm planning a hiking trip to Japan and need vegetarian food options."),
    MemoryMessage(role="assistant", content="Great choice! Japan has amazing hiking trails and excellent vegetarian cuisine."),
    MemoryMessage(role="user", content="I prefer nice hotels with good amenities, not too fancy but comfortable. All depends on the budget."),
    MemoryMessage(role="assistant", content="Noted! I'll remember your preference for comfortable mid-tier accommodations.")
]

working_memory_update = WorkingMemory(
    session_id=SESSION_ID_AUTO,
    namespace=NAMESPACE,
    messages=conversation,
    user_id=USER_ID_AUTO,
    # Strategy is already configured on the session
)

await client.put_working_memory(SESSION_ID_AUTO, working_memory_update)



WorkingMemoryResponse(messages=[MemoryMessage(role='user', content="I'm Nitin. I'm planning a hiking trip to Japan and need vegetarian food options.", id='01KGQ77RMQ6Z8EA9CRAY8FPVNP', created_at=datetime.datetime(2026, 2, 5, 15, 37, 52, 535148, tzinfo=TzInfo(0)), persisted_at=None, discrete_memory_extracted='f'), MemoryMessage(role='assistant', content='Great choice! Japan has amazing hiking trails and excellent vegetarian cuisine.', id='01KGQ77RMQ6Z8EA9CRAY8FPVNQ', created_at=datetime.datetime(2026, 2, 5, 15, 37, 52, 535182, tzinfo=TzInfo(0)), persisted_at=None, discrete_memory_extracted='f'), MemoryMessage(role='user', content='I prefer nice hotels with good amenities, not too fancy but comfortable. All depends on the budget.', id='01KGQ77RMQ6Z8EA9CRAY8FPVNR', created_at=datetime.datetime(2026, 2, 5, 15, 37, 52, 535201, tzinfo=TzInfo(0)), persisted_at=None, discrete_memory_extracted='f'), MemoryMessage(role='assistant', content="Noted! I'll remember your preference for comfortable mi

Conversation stored in working memory

The system will AUTOMATICALLY extract memories like:
  - 'Nitin is planning a hiking trip to Japan'
  - 'Nitin needs vegetarian food options'
  - 'Nitin prefers mid-range hotels with good amenities'
  - 'Nitin wants comfortable but not luxury accommodations'



In [25]:
# Step 3: Custom extraction strategy for specialized domains
# Use when you need specific information extracted

custom_strategy = MemoryStrategyConfig(
    strategy="custom",
    config={
        "custom_prompt": """Extract technical decisions and preferences from this conversation.

Focus on:
- Programming languages and frameworks mentioned
- Tool preferences
- Work schedule preferences
- Communication preferences

Format each as a clear, standalone statement.
Current datetime: {current_datetime}
Conversation: {message}"""
    }
)

# Create a new session with custom extraction
created, custom_memory = await client.get_or_create_working_memory(
    session_id="custom-extraction-demo",
    namespace=NAMESPACE,
    long_term_memory_strategy=custom_strategy
)

print(f"Session with custom extraction: {'created' if created else 'exists'}")
print(f"Custom prompt configured for specialized extraction")

Session with custom extraction: created
Custom prompt configured for specialized extraction


In [26]:
custom_memory

WorkingMemoryResponse(messages=[], memories=[], data={}, context=None, user_id=None, tokens=0, session_id='custom-extraction-demo', namespace='travel_agent', long_term_memory_strategy=MemoryStrategyConfig(strategy='custom', config={'custom_prompt': 'Extract technical decisions and preferences from this conversation.\n\nFocus on:\n- Programming languages and frameworks mentioned\n- Tool preferences\n- Work schedule preferences\n- Communication preferences\n\nFormat each as a clear, standalone statement.\nCurrent datetime: {current_datetime}\nConversation: {message}'}), ttl_seconds=None, last_accessed=datetime.datetime(2026, 2, 5, 15, 38, 56, 239184, tzinfo=TzInfo(0)), context_percentage_total_used=None, context_percentage_until_summarization=None, new_session=None, unsaved=None)

### Pattern 3 Summary

**Where memory operations happen:**
- `get_or_create_working_memory()` - Configure extraction strategy
- `put_working_memory()` - Store conversation, triggers extraction
- **Background worker** - Automatically extracts and stores memories

**Advantages:**
- Zero manual effort for memory storage
- Captures nuanced information automatically
- Continuous learning over time
- Configurable extraction strategies

**Disadvantages:**
- Less control over what gets stored
- Requires background worker running
- May extract irrelevant information
- Latency before memories are available

### How Topic and Entity Extraction Works

When memories are stored, the system can automatically extract **topics** and **entities** to improve search relevance.

#### Extraction Sources

| Source | When Used | How It Works |
|--------|-----------|--------------|
| **LLM-Provided** | When you explicitly set `topics` and `entities` in `ClientMemoryRecord` | Your values are used directly |
| **Auto-Extracted** | When `enable_topic_extraction=True` or `enable_ner=True` | System extracts from memory text |
| **Merged** | When both are present | Your values + auto-extracted values are combined |

#### Extraction Methods

The server supports two extraction backends (configured via environment variables):

**Topics** (`TOPIC_MODEL_SOURCE`):
- `"LLM"` (default): Uses the fast model to extract 3 topics per memory
- `"BERT"`: Uses BERTopic for local, faster extraction (requires PyTorch)

**Entities** (`NER_MODEL_SOURCE`):
- `"LLM"` (default): Uses the fast model to extract named entities (people, places, organizations)
- `"BERT"`: Uses a BERT-based NER model for local extraction (requires PyTorch)

#### How Topics/Entities Improve Search

When you search with `topics` or `entities` filters, the system:
1. First filters memories by the specified topics/entities (fast, exact match)
2. Then ranks by semantic similarity to your query (vector search)

```python
# Example: Search with topic filter
results = await client.search_long_term_memory(
    text="What are the user's preferences?",
    topics=Topics(any=["travel", "food"]),  # Must have at least one of these topics
    namespace={"eq": "travel_agent"}
)
```

#### Best Practices

1. **Provide topics when you know them**: If you're storing a memory about travel preferences, include `topics=["travel", "preferences"]`
2. **Let the system extract entities**: Named entities (people, places, companies) are often extracted accurately by the LLM
3. **Use topic filters for large memory stores**: When you have thousands of memories, topic filters significantly speed up search

### Background Extraction Configuration

The background extraction system uses a **trailing-edge debounce** mechanism to prevent constant re-extraction as new messages arrive.

#### Key Configuration

| Setting | Default | Description |
|---------|---------|-------------|
| `EXTRACTION_DEBOUNCE_SECONDS` | 30 | Seconds to wait after last message before extracting |
| `ENABLE_DISCRETE_MEMORY_EXTRACTION` | True | Whether to extract discrete memories from conversations |
| `ENABLE_TOPIC_EXTRACTION` | True | Whether to auto-extract topics |
| `ENABLE_NER` | True | Whether to auto-extract named entities |

#### How Debounce Works (especially important)

`EXTRACTION_DEBOUNCE_SECONDS` is a variable that controls how often background extraction tasks run. 
- Location: agent_memory_server/config.py
- Can be configured as an environment variable

```
Message 1 arrives → Schedule extraction for T+30s
Message 2 arrives (T+10s) → Reschedule extraction for T+40s
Message 3 arrives (T+25s) → Reschedule extraction for T+55s
No more messages → Extraction runs at T+55s
```

This ensures extraction happens after a "quiet period" in the conversation, not after every single message. Although sometimes we prefer that.


- **Too short** (e.g., 5s): Extraction runs mid-conversation, missing context
- **Too long** (e.g., 5min): Memories aren't available for a long time

Example of too short:
```
T+0s:  User: "I'm planning a trip to Japan next month."
T+5s:  ⚡ EXTRACTION RUNS (debounce expired)
       → Extracts: "User planning trip to Japan"
       
T+8s:  User: "I want to visit Tokyo and Kyoto."
T+13s: ⚡ EXTRACTION RUNS
       → Extracts: "User wants to visit Tokyo and Kyoto"
       
T+16s: User: "But I'm vegetarian and need wheelchair accessible hotels."
T+21s: ⚡ EXTRACTION RUNS
       → Extracts: "User is vegetarian, needs accessible hotels"
```

Alternatively at 30 secs:

```
T+0s:  User: "I'm planning a trip to Japan next month."
       → Schedule extraction for T+30s
       
T+8s:  User: "I want to visit Tokyo and Kyoto."
       → Reschedule extraction for T+38s
       
T+16s: User: "But I'm vegetarian and hotels with a pool."
       → Reschedule extraction for T+46s
       
T+46s: ⚡ EXTRACTION RUNS (30s of quiet)
       → Extracts: "User planning Japan trip to Tokyo and Kyoto, 
                    is vegetarian, needs hotels with a pool."
```

---

## Section 7: Combining Patterns

In most systems, it's useful to combine patterns, tools+background extraction for a little more control. 
```

### Example Files
- `examples/travel_agent.py` - Code-Driven with memory_prompt
- `examples/ai_tutor.py` - LLM-Driven with tool calling
- `examples/memory_prompt_agent.py` - Code-Driven with context enrichment

---

## Available APIs

The Agent Memory Client provides a comprehensive set of APIs for managing both working memory (session-scoped) and long-term memory (persistent). Here's a complete reference:

### API Flow Diagram

Here's how the APIs work together in a typical conversation:

```
┌─────────────────────────────────────────────────────────────────────────────────────────┐
│                           AGENT MEMORY API FLOW EXAMPLE                                  │
│                        User: "I'm planning a trip to Japan"                             │
└─────────────────────────────────────────────────────────────────────────────────────────┘

                                    ┌──────────────┐
                                    │     USER     │
                                    └──────┬───────┘
                                           │
                                           ▼
┌──────────────────────────────────────────────────────────────────────────────────────────┐
│  STEP 1: Get/Create Session                                                              │
│  ─────────────────────────────────────────────────────────────────────────────────────── │
│                                                                                          │
│    client.get_or_create_working_memory(session_id="nitin-travel-session")               │
│                           │                                                              │
│                           ▼                                                              │
│              ┌────────────────────────┐                                                  │
│              │    WORKING MEMORY      │  ← Session-scoped conversation state            │
│              │  ┌──────────────────┐  │                                                  │
│              │  │ messages: []     │  │                                                  │
│              │  │ memories: []     │  │                                                  │
│              │  │ context: null    │  │                                                  │
│              │  └──────────────────┘  │                                                  │
│              └────────────────────────┘                                                  │
└──────────────────────────────────────────────────────────────────────────────────────────┘
                                           │
                                           ▼
┌──────────────────────────────────────────────────────────────────────────────────────────┐
│  STEP 2: Hydrate Context with Memories                                                   │
│  ─────────────────────────────────────────────────────────────────────────────────────── │
│                                                                                          │
│    client.memory_prompt(query="I'm planning a trip to Japan", session_id=...)          │
│                           │                                                              │
│                           ├──────────────────────────────────────┐                       │
│                           ▼                                      ▼                       │
│              ┌────────────────────────┐         ┌────────────────────────────────┐      │
│              │    WORKING MEMORY      │         │       LONG-TERM MEMORY         │      │
│              │  (conversation history)│         │    (semantic vector search)    │      │
│              └───────────┬────────────┘         └───────────────┬────────────────┘      │
│                          │                                      │                        │
│                          │    ┌─────────────────────────────────┘                        │
│                          │    │  Finds: "User prefers vegetarian food"                  │
│                          │    │         "User likes hiking"                              │
│                          ▼    ▼                                                          │
│              ┌─────────────────────────────────────────────────────┐                    │
│              │              HYDRATED MESSAGES                       │                    │
│              │  ┌─────────────────────────────────────────────────┐│                    │
│              │  │ [SYSTEM] Long term memories:                    ││                    │
│              │  │   - User prefers vegetarian food                ││                    │
│              │  │   - User likes hiking                           ││                    │
│              │  ├─────────────────────────────────────────────────┤│                    │
│              │  │ [USER] I'm planning a trip to Japan             ││                    │
│              │  └─────────────────────────────────────────────────┘│                    │
│              └─────────────────────────────────────────────────────┘                    │
└──────────────────────────────────────────────────────────────────────────────────────────┘
                                           │
                                           ▼
┌──────────────────────────────────────────────────────────────────────────────────────────┐
│  STEP 3: Send to LLM (with optional tool schemas)                                        │
│  ─────────────────────────────────────────────────────────────────────────────────────── │
│                                                                                          │
│    tools = client.get_all_memory_tool_schemas()  ← Optional: Let LLM use memory tools  │
│                                                                                          │
│    openai.chat.completions.create(                                                       │
│        messages=hydrated_messages,                                                       │
│        tools=tools.to_list()          ← search_memory, add_memory, etc.                 │
│    )                                                                                     │
│                           │                                                              │
│                           ▼                                                              │
│              ┌─────────────────────────────────────────────────────┐                    │
│              │                    LLM RESPONSE                      │                    │
│              │  ┌─────────────────────────────────────────────────┐│                    │
│              │  │ Option A: Direct response                       ││                    │
│              │  │   "Based on your preferences, I recommend..."   ││                    │
│              │  ├─────────────────────────────────────────────────┤│                    │
│              │  │ Option B: Tool call                             ││                    │
│              │  │   tool_calls: [{name: "add_memory", args: ...}] ││                    │
│              │  └─────────────────────────────────────────────────┘│                    │
│              └─────────────────────────────────────────────────────┘                    │
└──────────────────────────────────────────────────────────────────────────────────────────┘
                                           │
                          ┌────────────────┴────────────────┐
                          ▼                                 ▼
┌─────────────────────────────────────────┐  ┌─────────────────────────────────────────────┐
│  STEP 4a: If LLM made tool calls        │  │  STEP 4b: Store conversation                │
│  ─────────────────────────────────────  │  │  ─────────────────────────────────────────  │
│                                         │  │                                             │
│  client.resolve_function_call(          │  │  client.put_working_memory(                 │
│      function_name="add_memory",        │  │      session_id,                            │
│      function_arguments={...},          │  │      WorkingMemory(messages=[               │
│      session_id=...                     │  │          {role: "user", content: "..."},    │
│  )                                      │  │          {role: "assistant", content: "..."}│
│           │                             │  │      ])                                     │
│           ▼                             │  │  )                                          │
│  ┌─────────────────────────┐            │  │           │                                 │
│  │ ToolCallResolutionResult│            │  │           ▼                                 │
│  │ ┌─────────────────────┐ │            │  │  ┌─────────────────────────────────────┐   │
│  │ │ success: true       │ │            │  │  │  Triggers background extraction     │   │
│  │ │ result: {...}       │ │            │  │  │  (after EXTRACTION_DEBOUNCE_SECONDS)│   │
│  │ │ formatted_response: │ │            │  │  │           │                         │   │
│  │ │   "Memory stored"   │ │            │  │  │           ▼                         │   │
│  │ └─────────────────────┘ │            │  │  │  ┌─────────────────────────────┐   │   │
│  └─────────────────────────┘            │  │  │  │ Extracted memories:         │   │   │
│                                         │  │  │  │ - "User planning Japan trip"│   │   │
│  Send formatted_response back to LLM    │  │  │  │ - "User interested in Tokyo"│   │   │
│  for final response                     │  │  │  └─────────────────────────────┘   │   │
│                                         │  │  └─────────────────────────────────────┘   │
└─────────────────────────────────────────┘  └─────────────────────────────────────────────┘


┌──────────────────────────────────────────────────────────────────────────────────────────┐
│  DATA FLOW SUMMARY                                                                       │
│  ─────────────────────────────────────────────────────────────────────────────────────── │
│                                                                                          │
│     USER MESSAGE                                                                         │
│          │                                                                               │
│          ▼                                                                               │
│    ┌───────────┐     memory_prompt()      ┌─────────────────┐                           │
│    │  Working  │◄─────────────────────────│   Long-Term     │                           │
│    │  Memory   │                          │   Memory        │                           │
│    │ (session) │──────────────────────────►│ (persistent)    │                           │
│    └───────────┘   background extraction  └─────────────────┘                           │
│          │              (debounced)              ▲                                       │
│          │                                       │                                       │
│          ▼                                       │                                       │
│    ┌───────────┐                                 │                                       │
│    │    LLM    │─────────────────────────────────┘                                       │
│    └───────────┘   create_long_term_memory()                                            │
│          │         (via tool call or code)                                              │
│          ▼                                                                               │
│    RESPONSE TO USER                                                                      │
│                                                                                          │
└──────────────────────────────────────────────────────────────────────────────────────────┘
```

### Working Memory Operations

| Method | Description | Notes |
|--------|-------------|-------|
| `get_working_memory(session_id)` | Get working memory for a session | Returns `None` if session doesn't exist. Use for checking if a session is active. |
| `get_or_create_working_memory(session_id)` | Get or create working memory | **Recommended for most use cases.** Returns `(created, memory)` tuple - `created` tells you if this is a new or returning user. |
| `put_working_memory(session_id, memory)` | Update/replace working memory for a session | Triggers background extraction to long-term memory (debounced). Call after each conversation turn. |
| `delete_working_memory(session_id)` | Delete working memory for a session | Use for explicit session cleanup. Sessions also auto-expire based on TTL. |
| `set_working_memory_data(session_id, data)` | Store arbitrary JSON data in working memory | Useful for storing structured state (e.g., form data, preferences) alongside messages. |
| `add_memories_to_working_memory(session_id, memories)` | Add structured memories to working memory | Adds to the `memories` list without replacing messages. Good for mid-conversation memory injection. |
| `list_sessions(namespace, limit, offset)` | List all sessions in a namespace | Useful for admin dashboards or finding active users. Supports pagination. |

### Long-Term Memory Operations

| Method | Description | Notes |
|--------|-------------|-------|
| `create_long_term_memory(memories)` | Create one or more long-term memories | Immediate storage (not debounced). Use for explicit "remember this" requests. Auto-generates embeddings. |
| `search_long_term_memory(text, ...)` | Semantic search with filters | Returns memories ranked by relevance. Filters: namespace, user_id, session_id, topics, entities, time range. |
| `get_long_term_memory(memory_id)` | Get a specific memory by ID | Use when you have the exact memory ID (e.g., from a previous search result). |
| `edit_long_term_memory(memory_id, updates)` | Edit an existing memory | Updates text and/or metadata. Re-generates embedding if text changes. |
| `delete_long_term_memories(memory_ids)` | Delete memories by ID | Permanent deletion. Use for user data deletion requests (GDPR, etc.). |
| `forget_long_term_memories(policy)` | Bulk delete based on policy | Policies: `before_date`, `after_date`, `inactive_for_days`. Good for automated cleanup. |

### Context Hydration

| Method | Description | Notes |
|--------|-------------|-------|
| `memory_prompt(query, session_id, ...)` | Get LLM-ready messages with relevant memories injected | **Core API for context enrichment.** Combines working memory + semantic search into ready-to-use messages. |
| `hydrate_memory_prompt(query, session_id, ...)` | Alias for `memory_prompt()` | Same functionality, alternative name for clarity. |

### LLM Tool Integration

| Method | Description | Notes |
|--------|-------------|-------|
| `get_all_memory_tool_schemas()` | Get OpenAI-compatible tool schemas | Returns schemas for `search_memory`, `add_memory`, etc. Pass to LLM's `tools` parameter. |
| `resolve_function_call(name, args, session_id)` | Execute a memory tool function call | Returns `ToolCallResolutionResult` with `result` (full data) and `formatted_response` (for LLM). |
| `resolve_tool_call(tool_call, session_id)` | Execute a tool call (auto-detects format) | Handles OpenAI, Anthropic, and other formats automatically. |
| `resolve_tool_calls(tool_calls, session_id)` | Execute multiple tool calls | Batch execution. Returns list of results in same order as input. |

### Summary Views (Advanced)

| Method | Description | Notes |
|--------|-------------|-------|
| `create_summary_view(request)` | Create a summary view configuration | Define filters and grouping for memory aggregation. Stored for reuse. |
| `list_summary_views()` | List all summary views | Returns all configured views in the namespace. |
| `get_summary_view(view_id)` | Get a summary view by ID | Retrieve configuration without running it. |
| `run_summary_view(view_id)` | Run a full summary view | Async task - returns task_id. Use `get_task()` to check completion. |
| `delete_summary_view(view_id)` | Delete a summary view | Removes the view configuration. |

### Utility Methods

| Method | Description | Notes |
|--------|-------------|-------|
| `health_check()` | Check server health | Returns server status. Use for monitoring and readiness probes. |
| `get_task(task_id)` | Get status of a background task | Check progress of async operations like `run_summary_view()`. Returns status, result, or error. |


---

## Section 8: Deep Dive - Working Memory Operations

Working memory stores the **current conversation state** for a session.

### Working Memory Summarization

When conversations grow long, the system automatically **summarizes older messages** to keep the context window manageable.

#### When Summarization Triggers

Summarization runs when the total tokens in working memory exceed **70% of the model's context window** (configurable via `SUMMARIZATION_THRESHOLD`).

```
┌─────────────────────────────────────────────────────────────────┐
│                    SUMMARIZATION TRIGGER                         │
├─────────────────────────────────────────────────────────────────┤
│                                                                  │
│   Model context window: 128,000 tokens (e.g., GPT-4)            │
│   Summarization threshold: 70% = 89,600 tokens                  │
│                                                                  │
│   Current messages: 95,000 tokens → EXCEEDS THRESHOLD           │
│          ↓                                                       │
│   Summarization runs automatically                               │
│                                                                  │
└─────────────────────────────────────────────────────────────────┘
```

#### How Summarization Works

1. **Calculate token usage**: Count tokens in all messages
2. **Identify messages to keep**: Keep ~40% of max tokens for recent messages
3. **Summarize older messages**: Generate a summary of older messages
4. **Store summary in `context` field**: The summary is stored in working memory's `context` field
5. **Remove summarized messages**: Older messages are deleted, keeping only recent ones

#### Token Allocation

| Model Context Size | Summary Max Tokens | Purpose |
|-------------------|-------------------|---------|
| < 10,000 tokens | 12.5% of context | Small models need proportionally more summary space |
| 10,000 - 50,000 tokens | 10% of context | Medium models |
| > 50,000 tokens | 5% of context | Large models can use smaller summaries |

#### Accessing the Summary

The summary is stored in the `context` field of working memory:

```python
wm = await client.get_working_memory(session_id, namespace="travel_agent")

if wm.context:
    print(f"Previous conversation summary: {wm.context}")
else:
    print("No summary yet (conversation is short)")
```

#### Progressive Summarization

When new messages arrive and another summarization is needed, the system uses **progressive summarization**:
- The new summary builds upon the previous summary
- This preserves important context from earlier in the conversation
- The prompt template is configurable via `PROGRESSIVE_SUMMARIZATION_PROMPT`

In [ ]:
# Working Memory Operations

# List all sessions in the travel_agent namespace
sessions = await client.list_sessions(namespace="travel_agent", limit=10)
print(f"Found {sessions.total} session(s) in 'travel_agent' namespace:")
for session in sessions.sessions[:5]:
    print(f"  - {session}")

In [59]:
# Get working memory for a session
# Note: Using the session from Pattern 1
session_to_check = "nitin-travel-session"

_, wm = await client.get_or_create_working_memory(
    session_id=session_to_check, 
    namespace="travel_agent"
)
print(f"Session: {wm.session_id}")
print(f"Messages: {len(wm.messages)}")
print(f"Memories: {len(wm.memories)}")
if wm.context:
    print(f"Context: {wm.context}")

Session: nitin-travel-session
Messages: 0
Memories: 0


In [60]:
# Store arbitrary JSON data in working memory
await client.set_working_memory_data(
    session_id="nitin-travel-session",
    data={
        "user_preferences": {"theme": "dark", "language": "en"},
        "trip_context": {"destination": "Japan", "dates": "March 2026"}
    },
    namespace="travel_agent"
)
print("Custom data stored in working memory!")

Custom data stored in working memory!


---

## Section 9: Deep Dive - Long-term Memory Operations

Long-term memory stores **persistent information** that survives across sessions.

In [61]:
# Search long-term memories with semantic search
import time
time.sleep(2)  # Wait for indexing

results = await client.search_long_term_memory(
    text="What are Nitin's travel preferences?",
    namespace={"eq": "travel_agent"},
    user_id={"eq": "nitin"},
    limit=5
)

print(f"Found {results.total} memories for user 'nitin':")
for memory in results.memories:
    relevance = (1 - memory.dist) * 100
    print(f"\n  [{relevance:.0f}%] {memory.text}")
    print(f"       Topics: {memory.topics}")

Found 5 memories for user 'nitin':

  [63%] User's name is Nitin and he is planning a trip to Tokyo.
       Topics: ['travel']

  [45%] User prefers comfortable mid-tier accommodations with good amenities while traveling.
       Topics: ['travel', 'accommodations', 'preferences']

  [41%] Prefers hotels with good amenities
       Topics: ['travel', 'preferences']

  [38%] Comfort travel - middle tier (not luxury, not budget)
       Topics: ['travel', 'preferences']

  [37%] User enjoys hiking and vegetarian food.
       Topics: ['travel', 'food', 'hiking']


In [62]:
# Search with filters
from agent_memory_client.filters import Topics, UserId

results = await client.search_long_term_memory(
    text="travel preferences",
    topics=Topics(any=["travel", "preferences"]),
    user_id=UserId(eq="nitin"),
    namespace={"eq": "travel_agent"},
    limit=10
)

print(f"Filtered search found {results.total} memories")
for mem in results.memories:
    print(f"  - {mem.text}")

Filtered search found 10 memories
  - Comfort travel - middle tier (not luxury, not budget)...
  - User prefers comfortable mid-tier accommodations with good a...
  - Extra leg room on flights (premium economy or exit row and b...


---

## Section 10: Memory Types & Contextual Grounding

### Semantic Memories
Facts, preferences, and general knowledge:
- "User prefers vegetarian food"
- "User works as a software engineer"

### Episodic Memories  
Events with specific times:
- "User visited Paris in March 2024"
- "User completed a marathon on January 15, 2025"

### Contextual Grounding
The system resolves pronouns and relative times:
- **Before:** "I went there yesterday"
- **After:** "User visited the coffee shop on February 4, 2026"

In [63]:
# Create an episodic memory with event date
from datetime import timedelta

episodic_memory = ClientMemoryRecord(
    text="User completed a 10-mile hike at Rocky Mountain National Park",
    memory_type=MemoryTypeEnum.EPISODIC,
    topics=["activities", "hiking", "achievements"],
    entities=["Rocky Mountain National Park"],
    event_date=datetime.now(timezone.utc) - timedelta(days=7),
    namespace=NAMESPACE
)

result = await client.create_long_term_memory([episodic_memory])
print(f"Created episodic memory: {result.status}")
print(f"Event date: {episodic_memory.event_date}")

Created episodic memory: ok
Event date: 2026-01-29 02:17:49.484219+00:00


---

## Section 11: Extraction Strategy Comparison

Different extraction strategies produce different results. Let's compare the **discrete** (default) strategy with a **custom** strategy using a travel agent scenario.

### The Scenario

A user provides detailed vehicle rental preferences with specific colors, features, and budget constraints.

In [ ]:
# Sample conversation with detailed vehicle preferences
vehicle_conversation = """
User: I'm looking to rent a car for my trip to Colorado next month.
Agent: I'd be happy to help! What type of vehicle are you looking for?
User: I definitely want an SUV - something like a Toyota 4Runner or Jeep Grand Cherokee.
      I prefer darker colors like charcoal gray or navy blue, nothing too flashy.
      Must have 4WD for the mountain roads, and I'd love heated seats since it'll be cold.
      My budget is around $75-100 per day, and I'll need it for 5 days starting March 15th.
Agent: Great choices! Any other preferences?
User: Yes, I need a roof rack for my ski equipment, and Apple CarPlay is a must.
      Also, I prefer to pick up from Denver airport rather than downtown.
"""

print("Sample conversation:")
print(vehicle_conversation)

### Strategy 1: Discrete (Default)

The discrete strategy extracts general semantic and episodic facts. It works well for most cases but may miss granular details.

In [ ]:
# What the DISCRETE strategy would extract (simulated output)
# In production, this happens automatically via background extraction

discrete_extraction_results = [
    {
        "type": "semantic",
        "text": "User prefers SUVs for car rentals",
        "topics": ["travel", "vehicles", "preferences"],
        "entities": ["SUV"]
    },
    {
        "type": "semantic",
        "text": "User prefers darker vehicle colors",
        "topics": ["preferences", "vehicles"],
        "entities": []
    },
    {
        "type": "episodic",
        "text": "User planning trip to Colorado in March 2026",
        "topics": ["travel", "trips"],
        "entities": ["Colorado", "March 2026"]
    },
    {
        "type": "semantic",
        "text": "User's car rental budget is $75-100 per day",
        "topics": ["budget", "travel"],
        "entities": []
    }
]

print("DISCRETE Strategy Results:")
print("=" * 50)
for i, mem in enumerate(discrete_extraction_results, 1):
    print(f"{i}. [{mem['type'].upper()}] {mem['text']}")
print(f"\nTotal memories extracted: {len(discrete_extraction_results)}")

### Strategy 2: Custom (Specialized for Vehicle Rentals)

A custom strategy with a specialized prompt can extract much more granular information.

In [ ]:
# Custom prompt for vehicle rental extraction
vehicle_rental_custom_prompt = '''
You are a vehicle rental preference extractor. Extract DETAILED vehicle preferences.

CURRENT CONTEXT:
Current date and time: {current_datetime}

Extract the following categories as separate memories:
1. VEHICLE TYPE: Specific makes/models mentioned
2. COLOR PREFERENCES: Exact colors mentioned
3. REQUIRED FEATURES: Must-have features (4WD, heated seats, etc.)
4. NICE-TO-HAVE FEATURES: Preferred but not required features
5. BUDGET: Exact budget range and duration
6. PICKUP/DROPOFF: Location and timing preferences
7. SPECIAL EQUIPMENT: Roof racks, child seats, etc.

Return JSON with format:
{{
    "memories": [
        {{"type": "semantic", "text": "...", "topics": [...], "entities": [...]}},
        ...
    ]
}}

Conversation:
{message}
'''

print("Custom prompt configured for granular vehicle preference extraction")

In [ ]:
# What the CUSTOM strategy would extract (simulated output)
custom_extraction_results = [
    {
        "type": "semantic",
        "text": "User prefers Toyota 4Runner or Jeep Grand Cherokee for rentals",
        "topics": ["vehicles", "preferences", "SUV"],
        "entities": ["Toyota 4Runner", "Jeep Grand Cherokee"]
    },
    {
        "type": "semantic",
        "text": "User prefers charcoal gray or navy blue vehicle colors",
        "topics": ["preferences", "colors"],
        "entities": ["charcoal gray", "navy blue"]
    },
    {
        "type": "semantic",
        "text": "User requires 4WD capability for mountain driving",
        "topics": ["requirements", "features"],
        "entities": ["4WD"]
    },
    {
        "type": "semantic",
        "text": "User requires heated seats in rental vehicles",
        "topics": ["requirements", "comfort"],
        "entities": ["heated seats"]
    },
    {
        "type": "semantic",
        "text": "User's rental budget is $75-100 per day for 5 days ($375-500 total)",
        "topics": ["budget", "pricing"],
        "entities": ["$75-100/day", "5 days"]
    },
    {
        "type": "episodic",
        "text": "User needs rental starting March 15, 2026 for Colorado trip",
        "topics": ["booking", "dates"],
        "entities": ["March 15, 2026", "Colorado"]
    },
    {
        "type": "semantic",
        "text": "User requires roof rack for ski equipment",
        "topics": ["equipment", "requirements"],
        "entities": ["roof rack", "ski equipment"]
    },
    {
        "type": "semantic",
        "text": "User requires Apple CarPlay in rental vehicle",
        "topics": ["technology", "requirements"],
        "entities": ["Apple CarPlay"]
    },
    {
        "type": "semantic",
        "text": "User prefers Denver airport pickup over downtown locations",
        "topics": ["pickup", "preferences"],
        "entities": ["Denver airport"]
    }
]

print("CUSTOM Strategy Results:")
print("=" * 50)
for i, mem in enumerate(custom_extraction_results, 1):
    print(f"{i}. [{mem['type'].upper()}] {mem['text']}")
print(f"\nTotal memories extracted: {len(custom_extraction_results)}")

### Side-by-Side Comparison

| Aspect | Discrete Strategy | Custom Strategy |
|--------|------------------|-----------------|
| **Memories extracted** | 4 | 9 |
| **Vehicle models** | Generic "SUVs" | "Toyota 4Runner", "Jeep Grand Cherokee" |
| **Colors** | "darker colors" | "charcoal gray", "navy blue" |
| **Features** | Not captured | 4WD, heated seats, Apple CarPlay, roof rack |
| **Budget** | "$75-100/day" | "$75-100/day for 5 days ($375-500 total)" |
| **Pickup location** | Not captured | "Denver airport" |

### When to Use Each Strategy

| Strategy | Best For |
|----------|----------|
| **Discrete** (default) | General conversations, broad preferences, most use cases |
| **Summary** | Long conversations that need condensing |
| **Preferences** | User settings and configuration extraction |
| **Custom** | Domain-specific extraction (travel, e-commerce, healthcare, etc.) |

### Configuring Custom Strategy in Production

```python
# Configure custom extraction when creating working memory
_, wm = await client.get_or_create_working_memory(
    session_id="vehicle-rental-session",
    namespace="travel_agent",
    memory_strategy=MemoryStrategyConfig(
        strategy="custom",
        custom_prompt=vehicle_rental_custom_prompt
    )
)
```

---

## Section 12: Production Considerations

### Authentication
```bash
# Production - NEVER disable auth
OAUTH2_ISSUER_URL=https://your-auth-provider.com
OAUTH2_AUDIENCE=your-api-audience
DISABLE_AUTH=false
```

### Background Task Worker
```bash
# Start API server
uv run agent-memory api

# Start task worker (separate process)
uv run agent-memory task-worker
```

### LLM Configuration
```bash
# OpenAI (default)
OPENAI_API_KEY=your-key
GENERATION_MODEL=gpt-4o-mini
EMBEDDING_MODEL=text-embedding-3-small

# Anthropic
ANTHROPIC_API_KEY=your-key
GENERATION_MODEL=claude-3-haiku-20240307
```

---

## Cleanup

Clean up demo data (optional).

In [64]:
# Cleanup demo data (optional)
cleanup = False  # Set to True to delete demo data

if cleanup:
    sessions_to_delete = [
        "nitin-travel-session",
        "nitin-llm-session", 
        "nitin-auto-session",
        "custom-extraction-demo"
    ]
    for sid in sessions_to_delete:
        try:
            await client.delete_working_memory(sid, namespace="travel_agent")
            print(f"Deleted session: {sid}")
        except Exception:
            pass
else:
    print("Cleanup skipped. Set cleanup=True to delete demo data.")

Cleanup skipped. Set cleanup=True to delete demo data.


---

## Summary

### The Three Integration Patterns

| Pattern | Key Methods | When to Use |
|---------|-------------|-------------|
| **Code-Driven** | `memory_prompt()`, `create_long_term_memory()` | Predictable behavior, workflows |
| **LLM-Driven** | `get_all_memory_tool_schemas()`, `resolve_function_call()` | Conversational agents |
| **Background** | `put_working_memory()` with `MemoryStrategyConfig` | Automatic learning |

### Key Takeaways

1. **Start with Code-Driven** for predictable behavior
2. **Add LLM-Driven** when you want conversational memory control
3. **Enable Background Extraction** for continuous learning
4. **Combine patterns** for production systems

### Resources
- API docs: `http://localhost:8000/docs`
- Examples: `examples/travel_agent.py`, `examples/ai_tutor.py`
- Docs: `docs/memory-integration-patterns.md`